# 20.7 Model serving and deployment

Serving is the controlled movement from artifact to traffic: load, warm, route, observe, and roll back before users pay for surprises.

Registries provide the artifact pointer and CI/CD provides gates before a serving system accepts it. This notebook turns those controls into routed load, warmup, capacity, and rollback calculations.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 2007
rng = np.random.default_rng(SEED)


def route_and_size(qps, capacity, canary_fraction, total_requests=40000):
    replicas = math.ceil(qps / capacity)
    canary_requests = canary_fraction * total_requests
    return replicas, canary_requests


def serving_ladder(seed=SEED):
    local_rng = np.random.default_rng(seed)
    specs = [
        {"rung": "D1", "name": "tiny canary trace", "n": 40, "qps": 25, "capacity": 150, "canary": 0.05, "cold": 1},
        {"rung": "D2", "name": "10k stable requests", "n": 10000, "qps": 120, "capacity": 150, "canary": 0.05, "cold": 0},
        {"rung": "D3", "name": "cold starts and peak qps", "n": 30000, "qps": 900, "capacity": 150, "canary": 0.10, "cold": 6},
        {"rung": "D4", "name": "warm deployment trace", "n": 60000, "qps": 700, "capacity": 150, "canary": 0.25, "cold": 0},
        {"rung": "D5", "name": "canary rollback simulation", "n": 160000, "qps": 1200, "capacity": 150, "canary": 0.50, "cold": 10},
    ]
    workloads = []
    for spec in specs:
        n = spec["n"]
        t = np.arange(n) / max(spec["qps"], 1)
        wave = 1.0 + 0.25 * np.sin(2.0 * np.pi * t / max(t[-1], 1.0))
        load = spec["qps"] * wave * local_rng.lognormal(mean=0.0, sigma=0.08, size=n)
        routed_candidate = local_rng.random(n) < spec["canary"]
        workloads.append({**spec, "load": load, "candidate": routed_candidate})
    return workloads


def simulate_serving(workload, warmup=True):
    replicas = math.ceil(workload["qps"] / workload["capacity"])
    load_per_replica = workload["load"] / max(replicas, 1)
    utilization = load_per_replica / workload["capacity"]
    base_ms = 38.0 + 18.0 * workload["candidate"]
    queue_ms = np.maximum(utilization - 0.70, 0.0) * 120.0
    cold_penalty = np.zeros(workload["n"])
    if not warmup:
        cold_count = min(workload["cold"], workload["n"])
        cold_penalty[:cold_count] = 30000.0
    jitter = rng.normal(0.0, 4.0, size=workload["n"])
    latency = np.maximum(base_ms + queue_ms + cold_penalty + jitter, 1.0)
    served_qps = min(workload["qps"], replicas * workload["capacity"])
    error_rate = 0.002 + 0.004 * (np.mean(utilization) > 0.90) + 0.003 * np.mean(workload["candidate"])
    return {"latency_ms": latency, "replicas": replicas, "served_qps": served_qps, "error_rate": error_rate}


## The concept, built once: route and size

Serving capacity uses $r=\lceil q/c\rceil$, while a canary receives $n_{canary}=\alpha n$. These are tiny formulas, but they decide whether users hit a warmed model or a cold surprise.

In [ ]:

canary_fraction = 0.05
total_requests = 40000
canary_requests = canary_fraction * total_requests
qps = 900
capacity = 150
replicas = math.ceil(qps / capacity)

assert canary_requests == 2000
assert replicas == 6
print("canary requests", canary_requests)
print("replicas required", replicas)


Warmup is also arithmetic: compiling for $30$ seconds before traffic avoids adding $30$ seconds to the first request. Shadow mode mirrors $10{,}000$ evaluations with $0\%$ user exposure; an error rate $0.003$ versus limit $0.005$ leaves $0.002$ margin.

In [ ]:

warmup_seconds = 30
shadow_evaluations = 10000
shadow_served_responses = 0
error_rate = 0.003
limit = 0.005
margin = limit - error_rate

assert warmup_seconds == 30
assert shadow_evaluations == 10000
assert shadow_served_responses == 0
assert round(margin, 3) == 0.002
print("warmup seconds before routing", warmup_seconds)
print("shadow evaluations", shadow_evaluations)
print("blue-green error margin", margin)


## The dataset ladder
D1-D5 move from a tiny canary trace to a production canary/rollback simulation.

In [ ]:

workloads = serving_ladder()
for workload in workloads:
    preview = pd.DataFrame({
        "candidate_route": workload["candidate"][:8],
        "load_qps": np.round(workload["load"][:8], 1),
    })
    print(workload["rung"], workload["name"], "n=", workload["n"], "qps=", workload["qps"])
    print(preview)


In [ ]:

rows = []
outputs = []
for workload in workloads:
    result = simulate_serving(workload, warmup=True)
    outputs.append((workload, result))
    p95 = float(np.percentile(result["latency_ms"], 95))
    throughput_ratio = result["served_qps"] / workload["qps"]
    rows.append({
        "rung": workload["rung"],
        "workload": workload["name"],
        "metric_served_throughput_qps": float(result["served_qps"]),
        "throughput_ratio": float(throughput_ratio),
        "p95_latency_ms": p95,
        "replicas": result["replicas"],
        "error_rate": float(result["error_rate"]),
    })
metrics = pd.DataFrame(rows)
print(metrics.to_string(index=False))


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for i, (workload, result) in enumerate(outputs):
    axes[0, i].hist(result["latency_ms"], bins=35, color="#4c78a8")
    axes[0, i].set_title(workload["rung"])
    axes[0, i].set_xlabel("latency ms")
    axes[0, i].set_ylabel("requests")
    panel_values = [np.percentile(result["latency_ms"], 95), result["served_qps"], result["replicas"]]
    axes[1, i].bar(["p95", "qps", "replicas"], panel_values, color=["#f58518", "#54a24b", "#b279a2"])
    axes[1, i].set_title("operational panel")
summary_x = np.arange(len(metrics))
fig2, ax = plt.subplots(figsize=(7, 4))
ax.plot(summary_x, metrics["metric_served_throughput_qps"], marker="o", label="served qps")
ax.plot(summary_x, metrics["p95_latency_ms"], marker="s", label="p95 latency")
ax.set_xticks(summary_x)
ax.set_xticklabels(metrics["rung"])
ax.set_title("throughput and p95 vs routed load")
ax.set_xlabel("D1 to D5 load")
ax.legend()
plt.show()


## Pitfall on D5: cold-starting under traffic
If the candidate is routed before warmup, the first users pay the compile cost. Warming before routing moves that cost out of the user path.

In [ ]:

d5 = workloads[-1]
cold_result = simulate_serving(d5, warmup=False)
warm_result = simulate_serving(d5, warmup=True)
wrong_max = float(np.max(cold_result["latency_ms"][:20]))
fixed_max = float(np.max(warm_result["latency_ms"][:20]))
print("first-user cold-start ms", wrong_max)
print("first-user after warmup ms", fixed_max)
print("spike avoided ms", wrong_max - fixed_max)


## Evaluate it + practice

- Metric: served throughput qps, with p95 latency and error-rate guardrails.
- No-skill baseline: route 100% to the old blue model forever.
- Cheap sanity check: replicas should equal ceil(qps / capacity).
- Ablation: disable warmup or autoscaling and watch p95 or throughput degrade.
- Failure signals: cold spikes, shadow confusion, rollback margin below zero, or capacity below peak load.

Practice: Change canary_fraction on D4 and recompute candidate traffic.

Practice: Add an error-rate rollback rule to D5.

Practice: Compare shadow traffic and canary traffic on the same plot.